# Held-out library-adequacy diagnostic (Ex 20.1–20.3)

**Paper:** *Gaeta–Lie Neural SDEs: Symmetry-Regularized Learning of Stochastic Dynamics* (NeurIPS 2026) — Section 4.2, Appendix C.5.

Spectral Stage II with the held-out adequacy test (validation residual, random-subspace comparison, train–validation agreement, bootstrap stability). Compares the complete R = 208 library with a reduced R = 42 control library and checks that abstention returns Stage I unchanged.

**Pipeline naming used in the code:** Stage I = baseline neural SDE surrogate; Stage II = symmetry discovery over the basis library; Stage F = Stage III (symmetry-regularized refinement).

Run the cells top to bottom. All configuration lives in `CONFIG` in the first code cell. See the [repository README](../../README.md) for installation and the full experiment index.

## Cell 1 — Imports and complete E6 configuration

In [ ]:
# ============================================================================
# Cell 1 — Imports and complete E6 configuration
# ============================================================================

from jax import config as jax_config
jax_config.update("jax_enable_x64", False)

import copy
import gc
import math
import pickle
import time
from pathlib import Path

import jax
import jax.numpy as jnp
import numpy as np

print("JAX version:", jax.__version__)
print("JAX devices:", jax.devices())


CONFIG = {
    # ------------------------------------------------------------------
    # Experiment scope
    # ------------------------------------------------------------------
    "SEED": 42,
    "E6_EXAMPLES": [
        "ex20_1_phase_difference_ring",
        "ex20_2_tamed_lorenz_chain",
        "ex20_3_cyclic_reaction_ring",
    ],
    "STATE_DIM": 20,
    "N_GENERATORS_MAX": 15,

    # Set True only for a short plumbing test. Reported runs must use False.
    "DEBUG_FAST": False,

    # ------------------------------------------------------------------
    # Data
    # The manuscript data-efficient rule is N_traj = 2d = 40.
    # E6 needs a validation split for the adequacy gate, so trajectories are
    # split 28/6/6 instead of mixing adjacent increments across splits.
    # ------------------------------------------------------------------
    "N_TRAJECTORIES": 40,
    "TRAIN_FRACTION": 0.70,
    "VALIDATION_FRACTION": 0.15,
    "TEST_FRACTION": 0.15,

    # ------------------------------------------------------------------
    # Neural SDE surrogate: manuscript 20D architecture
    # ------------------------------------------------------------------
    "MLP_HIDDEN_DIM": 96,
    "MLP_DEPTH": 3,
    "SIGMA_EPS": 1.0e-4,
    "GLOBAL_GRAD_CLIP": 10.0,

    # ------------------------------------------------------------------
    # Stage I: manuscript 20D settings
    # ------------------------------------------------------------------
    "BASELINE_EPOCHS": 2000,
    "BASELINE_BATCH_SIZE": 512,
    "BASELINE_LR": 2.0e-3,
    "BASELINE_EVAL_EVERY": 100,
    "PRINT_EVERY": 250,

    # ------------------------------------------------------------------
    # Complete and reduced libraries
    # ------------------------------------------------------------------
    # The complete grid reproduces Appendix C.3 exactly and gives R=208.
    "COMPLETE_MU_LIST": [-2.0, -0.5, 0.5, 2.0],

    # Side sensitivity analysis. Each candidate creates a 145-template
    # complete-structure library: all 124 non-exponential complete-library
    # templates plus 20 exponential translations and one projective template.
    "MU_SWEEP_LIST": [-3.0, -2.0, -1.0, -0.5, 0.5, 1.0, 2.0, 3.0],
    "BASIS_EXP_CLIP": 8.0,

    # ------------------------------------------------------------------
    # Generalized determining-operator diagnostic
    # ------------------------------------------------------------------
    "E6_SPECTRAL_TRAIN_POINTS": 192,
    "E6_SPECTRAL_VALIDATION_POINTS": 192,
    "E6_ORACLE_POINTS": 192,
    "E6_OPERATOR_CHUNK_SIZE": 8,
    # Rank is computed after empirical unit-norm normalization of every
    # generator template. At this tolerance, the exact R=208 library has
    # the expected generic function-space rank 167.
    "E6_GRAM_REL_TOL": 1.0e-7,
    "E6_GRAM_ABS_TOL": 1.0e-10,
    "E6_EXPECTED_COMPLETE_RANK": 167,
    "E6_EXPECTED_SINGLE_MU_RANK": 104,
    "E6_EXPECTED_CONTROL_RANK": 42,
    "E6_ENFORCE_LIBRARY_RANK": True,
    "E6_MATRIX_RIDGE": 1.0e-10,
    "E6_N_RANDOM_SUBSPACES": 256,
    "E6_RANDOM_NULL_QUANTILE": 0.01,
    "E6_BOOTSTRAP_REPEATS": 5,
    "E6_N_SPECTRUM_VALUES_TO_PRINT": 20,
    "E6_NORMALIZE_OPERATOR_RESIDUAL": True,

    # Validation-only adequacy gate. We discover up to 15 directions, then
    # select the largest accepted prefix m_hat in {0,...,15}. No coefficient
    # truth or test MSE enters this selection.
    "E6_GATE_MAX_VALIDATION_SCORE": 1.5e-1,
    "E6_GATE_MAX_SCORE_TO_NULL": 0.50,
    "E6_GATE_MAX_SCORE_TO_MEDIAN": 0.08,
    "E6_GATE_MAX_TRAIN_VAL_INFLATION": 5.0,
    "E6_GATE_MAX_BOOTSTRAP_MAX_ANGLE_DEG": 30.0,

    # ------------------------------------------------------------------
    # Stage III: manuscript 20D settings
    # ------------------------------------------------------------------
    "STAGE_F_EPOCHS": 1200,
    "STAGE_F_BATCH_SIZE": 512,
    "STAGE_F_LR": 1.0e-3,
    "STAGE_F_LAMBDA_SYM_DET": 0.05,
    "STAGE_F_N_DATA_SYM_POINTS": 128,
    "STAGE_F_N_EXTRA_SYM_POINTS": 512,
    "STAGE_F_EVAL_EVERY": 100,

    # ------------------------------------------------------------------
    # Evaluation
    # ------------------------------------------------------------------
    "N_EVAL_POINTS": 5000,
    "N_RESIDUAL_EVAL_POINTS": 256,
    "E6_MAX_ABSTENTION_IDENTITY_ERROR": 1.0e-12,
    "E6_MAX_MATERIAL_DEGRADATION_PCT": 5.0,

    # Optional numerical checkpointing. Parameter trees are not saved.
    "E6_RESULTS_PATH": "./E6_20D_one_seed_results.pkl",
}


if CONFIG["DEBUG_FAST"]:
    CONFIG.update({
        "E6_EXAMPLES": ["ex20_1_phase_difference_ring"],
        "N_TRAJECTORIES": 12,
        "BASELINE_EPOCHS": 5,
        "BASELINE_EVAL_EVERY": 1,
        "STAGE_F_EPOCHS": 3,
        "STAGE_F_EVAL_EVERY": 1,
        "E6_SPECTRAL_TRAIN_POINTS": 12,
        "E6_SPECTRAL_VALIDATION_POINTS": 12,
        "E6_ORACLE_POINTS": 12,
        "E6_OPERATOR_CHUNK_SIZE": 2,
        "E6_N_RANDOM_SUBSPACES": 12,
        "E6_BOOTSTRAP_REPEATS": 2,
        "STAGE_F_N_DATA_SYM_POINTS": 8,
        "STAGE_F_N_EXTRA_SYM_POINTS": 8,
        "N_EVAL_POINTS": 32,
        "N_RESIDUAL_EVAL_POINTS": 12,
        "PRINT_EVERY": 1,
    })


D = int(CONFIG["STATE_DIM"])
Q = D + 1
M_MAX = int(CONFIG["N_GENERATORS_MAX"])

if D != 20:
    raise ValueError("This notebook is specifically designed for d=20.")
if not (1 <= M_MAX <= D + 2):
    raise ValueError(f"N_GENERATORS_MAX must lie in [1, d+2]; got {M_MAX}.")
if int(CONFIG["E6_BOOTSTRAP_REPEATS"]) != (2 if CONFIG["DEBUG_FAST"] else 5):
    raise ValueError("The full E6 run must use exactly five bootstrap rediscoveries.")

split_sum = (
    CONFIG["TRAIN_FRACTION"]
    + CONFIG["VALIDATION_FRACTION"]
    + CONFIG["TEST_FRACTION"]
)
if not np.isclose(split_sum, 1.0):
    raise ValueError(f"Train/validation/test fractions sum to {split_sum}, not 1.")

print("\nE6 20D configuration")
print("  seed                    :", CONFIG["SEED"])
print("  examples                :", CONFIG["E6_EXAMPLES"])
print("  state dimension         :", D)
print("  maximum generators m_max:", M_MAX)
print("  trajectories per example:", CONFIG["N_TRAJECTORIES"])
print("  bootstrap repeats       :", CONFIG["E6_BOOTSTRAP_REPEATS"])
print("  DEBUG_FAST              :", CONFIG["DEBUG_FAST"])

## Cell 2 — Three manuscript 20D SDEs and trajectory-level splits

In [ ]:
# ============================================================================
# Cell 2 — Three manuscript 20D SDEs and trajectory-level splits
# ============================================================================

_THETA = 2.0 * jnp.pi * jnp.arange(D, dtype=jnp.float32) / float(D)


def drift_ex20_1(t, x):
    """Appendix A.3, Ex. 20.1: phase-difference ring."""
    xp1 = jnp.roll(x, -1, axis=-1)
    xm1 = jnp.roll(x, 1, axis=-1)
    xm2 = jnp.roll(x, 2, axis=-1)
    return (
        0.75 * jnp.sin(xp1 - x)
        + 0.45 * jnp.sin(xm1 - x)
        + 0.20 * jnp.sin(2.0 * (xp1 - x))
        + 0.15 * jnp.sin(xm2 - x)
    )


def sigma_ex20_1(t, x):
    del t
    sigma = 0.28 + 0.04 * jnp.sin(_THETA + 0.3)
    return jnp.broadcast_to(sigma, x.shape)


def drift_ex20_2(t, x):
    """Appendix A.3, Ex. 20.2: tamed Lorenz chain."""
    xp1 = jnp.roll(x, -1, axis=-1)
    xm1 = jnp.roll(x, 1, axis=-1)
    numerator = (
        2.8 * (xp1 - x)
        + 0.9 * xm1 * (1.5 - 0.15 * xp1)
        - 0.85 * x
        + 0.08 * jnp.sin(1.2 * t + _THETA)
    )
    denominator = 1.0 + 0.04 * jnp.sum(x * x, axis=-1, keepdims=True)
    return numerator / denominator


def sigma_ex20_2(t, x):
    del t
    base = 0.18 + 0.025 * jnp.sin(_THETA + 0.2)
    amplitude = 0.035 + 0.008 * jnp.sin(_THETA + 1.2)
    return base + amplitude * jnp.tanh(x) ** 2


def drift_ex20_3(t, x):
    """Appendix A.3, Ex. 20.3: cyclic reaction ring."""
    xp1 = jnp.roll(x, -1, axis=-1)
    xm1 = jnp.roll(x, 1, axis=-1)
    xp2 = jnp.roll(x, -2, axis=-1)
    return (
        -(0.50 + 0.10 * jnp.sin(_THETA)) * x
        + 0.72 * jnp.tanh(xp1)
        - 0.30 * jnp.tanh(xm1)
        + 0.18 * jnp.tanh(xp2)
        + 0.08 * jnp.sin(t + _THETA)
    )


def sigma_ex20_3(t, x):
    del t
    base = 0.24 + 0.020 * jnp.sin(_THETA + 0.5)
    amplitude = 0.045 + 0.010 * jnp.sin(_THETA + 1.1)
    return base + amplitude * jnp.sin(x) ** 2


# The paper specifies the equations and common 20D training protocol but does
# not enumerate separate support boxes in Appendix A. These fixed boxes cover
# the simulated trajectory clouds and are used consistently for Stage-II
# diagnostics, Stage-III collocation, and coefficient evaluation.
SDE_EXAMPLES = {
    "ex20_1_phase_difference_ring": {
        "label": "Ex. 20.1 phase-difference ring",
        "drift_fn": drift_ex20_1,
        "sigma_fn": sigma_ex20_1,
        "t_final": 2.0,
        "dt": 0.01,
        "x0_mean": 0.0,
        "x0_std": 0.75,
        "input_t_scale": 2.0,
        "input_x_scale": 3.0,
        "x_box": (-3.0, 3.0),
    },
    "ex20_2_tamed_lorenz_chain": {
        "label": "Ex. 20.2 tamed Lorenz chain",
        "drift_fn": drift_ex20_2,
        "sigma_fn": sigma_ex20_2,
        "t_final": 2.0,
        "dt": 0.01,
        "x0_mean": 0.0,
        "x0_std": 0.80,
        "input_t_scale": 2.0,
        "input_x_scale": 4.0,
        "x_box": (-4.0, 4.0),
    },
    "ex20_3_cyclic_reaction_ring": {
        "label": "Ex. 20.3 cyclic reaction ring",
        "drift_fn": drift_ex20_3,
        "sigma_fn": sigma_ex20_3,
        "t_final": 2.0,
        "dt": 0.01,
        "x0_mean": 0.0,
        "x0_std": 0.80,
        "input_t_scale": 2.0,
        "input_x_scale": 3.0,
        "x_box": (-3.0, 3.0),
    },
}


def make_simulator(spec):
    drift_fn = spec["drift_fn"]
    sigma_fn = spec["sigma_fn"]
    t_final = float(spec["t_final"])
    dt = float(spec["dt"])
    n_steps = int(round(t_final / dt))
    n_trajectories = int(CONFIG["N_TRAJECTORIES"])
    x0_mean = float(spec["x0_mean"])
    x0_std = float(spec["x0_std"])

    @jax.jit
    def simulate(key):
        key_x0, key_noise = jax.random.split(key)
        x0 = x0_mean + x0_std * jax.random.normal(key_x0, (n_trajectories, D))
        noise = jax.random.normal(key_noise, (n_steps, n_trajectories, D))
        times = jnp.arange(n_steps + 1, dtype=jnp.float32) * dt
        sqrt_dt = jnp.sqrt(jnp.asarray(dt, dtype=jnp.float32))

        def step(x, inp):
            t, zeta = inp
            f = drift_fn(t, x)
            sigma = sigma_fn(t, x)
            x_next = x + dt * f + sqrt_dt * sigma * zeta
            return x_next, x_next

        _, tail = jax.lax.scan(step, x0, (times[:-1], noise))
        paths = jnp.concatenate([x0[None, :, :], tail], axis=0)
        return times, jnp.transpose(paths, (1, 0, 2))

    return simulate


def make_increment_dataset(times, paths):
    n_paths, n_times, dim = paths.shape
    n_steps = n_times - 1
    t0 = jnp.broadcast_to(times[:-1][None, :], (n_paths, n_steps)).reshape(-1)
    x0 = paths[:, :-1, :].reshape(-1, dim)
    dx = (paths[:, 1:, :] - paths[:, :-1, :]).reshape(-1, dim)
    dt = jnp.full((x0.shape[0],), times[1] - times[0], dtype=jnp.float32)
    return t0, x0, dt, dx


def split_paths_by_trajectory(key, paths):
    n_paths = int(paths.shape[0])
    permutation = np.asarray(jax.random.permutation(key, n_paths))

    n_train = int(round(CONFIG["TRAIN_FRACTION"] * n_paths))
    n_val = int(round(CONFIG["VALIDATION_FRACTION"] * n_paths))
    n_train = max(1, min(n_train, n_paths - 2))
    n_val = max(1, min(n_val, n_paths - n_train - 1))

    train_idx = permutation[:n_train]
    val_idx = permutation[n_train:n_train + n_val]
    test_idx = permutation[n_train + n_val:]
    return paths[train_idx], paths[val_idx], paths[test_idx]


def make_example_data(example_name, seed):
    spec = SDE_EXAMPLES[example_name]
    key = jax.random.PRNGKey(int(seed) + 1000 * list(SDE_EXAMPLES).index(example_name))
    key_sim, key_split = jax.random.split(key)
    times, paths = make_simulator(spec)(key_sim)
    train_paths, val_paths, test_paths = split_paths_by_trajectory(key_split, paths)
    return {
        "times": times,
        "paths": paths,
        "train_paths": train_paths,
        "validation_paths": val_paths,
        "test_paths": test_paths,
        "train": make_increment_dataset(times, train_paths),
        "validation": make_increment_dataset(times, val_paths),
        "test": make_increment_dataset(times, test_paths),
    }


def sample_batch(key, dataset, batch_size):
    n = int(dataset[0].shape[0])
    idx = jax.random.randint(key, (int(batch_size),), 0, n)
    return tuple(arr[idx] for arr in dataset)


def sample_tx_from_dataset(key, dataset, n_points):
    t0, x0, _, _ = dataset
    idx = jax.random.randint(key, (int(n_points),), 0, int(t0.shape[0]))
    return jnp.concatenate([t0[idx, None], x0[idx]], axis=1)


def sample_tx_box(key, n_points, spec):
    key_t, key_x = jax.random.split(key)
    low, high = spec["x_box"]
    t = jax.random.uniform(
        key_t,
        (int(n_points), 1),
        minval=0.0,
        maxval=float(spec["t_final"]),
    )
    x = jax.random.uniform(
        key_x,
        (int(n_points), D),
        minval=float(low),
        maxval=float(high),
    )
    return jnp.concatenate([t, x], axis=1)


for _name in CONFIG["E6_EXAMPLES"]:
    _spec = SDE_EXAMPLES[_name]
    _probe = jnp.zeros((4, D), dtype=jnp.float32)
    _f = _spec["drift_fn"](jnp.asarray(0.37, dtype=jnp.float32), _probe)
    _s = _spec["sigma_fn"](jnp.asarray(0.37, dtype=jnp.float32), _probe)
    if _f.shape != (4, D) or _s.shape != (4, D):
        raise RuntimeError(f"Bad coefficient shape for {_name}: {_f.shape}, {_s.shape}")
    if float(jnp.min(_s)) <= 0.0:
        raise RuntimeError(f"Nonpositive diffusion in {_name}.")
    print(f"{_spec['label']}: coefficient probe OK; min sigma={float(jnp.min(_s)):.4f}")

## Cell 3 — Neural SDE, optimizer, and Stage-I training

In [ ]:
# ============================================================================
# Cell 3 — Neural SDE, optimizer, and Stage-I training
# ============================================================================


def glorot_uniform(key, in_dim, out_dim):
    limit = jnp.sqrt(6.0 / float(in_dim + out_dim))
    return jax.random.uniform(key, (in_dim, out_dim), minval=-limit, maxval=limit)


class MLP:
    def __init__(self, in_dim, hidden_dim, depth, out_dim):
        self.in_dim = int(in_dim)
        self.hidden_dim = int(hidden_dim)
        self.depth = int(depth)
        self.out_dim = int(out_dim)

    def init(self, key):
        sizes = [self.in_dim] + [self.hidden_dim] * self.depth + [self.out_dim]
        keys = jax.random.split(key, len(sizes) - 1)
        return [
            {
                "W": glorot_uniform(k, din, dout),
                "b": jnp.zeros((dout,), dtype=jnp.float32),
            }
            for k, din, dout in zip(keys, sizes[:-1], sizes[1:])
        ]

    def apply(self, params, z):
        h = z
        for layer in params[:-1]:
            h = jnp.tanh(h @ layer["W"] + layer["b"])
        return h @ params[-1]["W"] + params[-1]["b"]


surrogate_model = MLP(
    in_dim=Q,
    hidden_dim=CONFIG["MLP_HIDDEN_DIM"],
    depth=CONFIG["MLP_DEPTH"],
    out_dim=2 * D,
)


def normalize_input(t, x, t_scale, x_scale):
    return jnp.concatenate(
        [jnp.asarray([t / t_scale], dtype=jnp.float32), x / x_scale],
        axis=0,
    )


def surrogate_f_sigma(params, t, x, t_scale, x_scale):
    raw = surrogate_model.apply(params, normalize_input(t, x, t_scale, x_scale))
    f = raw[:D]
    sigma = jax.nn.softplus(raw[D:]) + CONFIG["SIGMA_EPS"]
    return f, sigma


def data_nll_loss(params, batch, t_scale, x_scale):
    t0, x0, dt, dx = batch
    f, sigma = jax.vmap(
        lambda tt, xx: surrogate_f_sigma(params, tt, xx, t_scale, x_scale)
    )(t0, x0)
    mean = f * dt[:, None]
    variance = sigma**2 * dt[:, None] + 1.0e-8
    nll = 0.5 * ((dx - mean) ** 2 / variance + jnp.log(2.0 * jnp.pi * variance))
    return jnp.mean(jnp.sum(nll, axis=-1))


def tree_copy(tree):
    return jax.tree_util.tree_map(lambda x: jnp.array(x, copy=True), tree)


def tree_l2_norm(tree):
    leaves = jax.tree_util.tree_leaves(tree)
    return jnp.sqrt(sum(jnp.sum(x * x) for x in leaves) + 1.0e-30)


def tree_all_finite(tree):
    leaves = jax.tree_util.tree_leaves(tree)
    return jnp.all(jnp.stack([jnp.all(jnp.isfinite(x)) for x in leaves]))


def tree_max_abs_diff(a, b):
    leaves = jax.tree_util.tree_leaves(
        jax.tree_util.tree_map(lambda x, y: jnp.max(jnp.abs(x - y)), a, b)
    )
    return float(max(float(x) for x in leaves)) if leaves else 0.0


def clip_grad_tree(grads, max_norm):
    norm = tree_l2_norm(grads)
    scale = jnp.minimum(1.0, float(max_norm) / (norm + 1.0e-12))
    return jax.tree_util.tree_map(lambda g: scale * g, grads), norm


def adam_init(params):
    return {
        "m": jax.tree_util.tree_map(jnp.zeros_like, params),
        "v": jax.tree_util.tree_map(jnp.zeros_like, params),
        "t": jnp.array(0, dtype=jnp.int32),
    }


def adam_update(params, grads, state, lr, beta1=0.9, beta2=0.999, eps=1.0e-8):
    t_new = state["t"] + 1
    m_new = jax.tree_util.tree_map(
        lambda m, g: beta1 * m + (1.0 - beta1) * g, state["m"], grads
    )
    v_new = jax.tree_util.tree_map(
        lambda v, g: beta2 * v + (1.0 - beta2) * g * g, state["v"], grads
    )
    m_hat = jax.tree_util.tree_map(lambda m: m / (1.0 - beta1**t_new), m_new)
    v_hat = jax.tree_util.tree_map(lambda v: v / (1.0 - beta2**t_new), v_new)
    new_params = jax.tree_util.tree_map(
        lambda p, m, v: p - float(lr) * m / (jnp.sqrt(v) + eps),
        params,
        m_hat,
        v_hat,
    )
    return new_params, {"m": m_new, "v": v_new, "t": t_new}


@jax.jit
def baseline_train_step(params, opt_state, batch, t_scale, x_scale):
    loss, grads = jax.value_and_grad(data_nll_loss)(
        params, batch, t_scale, x_scale
    )
    grads, grad_norm = clip_grad_tree(grads, CONFIG["GLOBAL_GRAD_CLIP"])
    new_params, new_state = adam_update(
        params, grads, opt_state, CONFIG["BASELINE_LR"]
    )
    finite = jnp.logical_and(jnp.isfinite(loss), tree_all_finite(new_params))
    kept_params = jax.tree_util.tree_map(
        lambda new, old: jnp.where(finite, new, old), new_params, params
    )
    kept_state = jax.tree_util.tree_map(
        lambda new, old: jnp.where(finite, new, old), new_state, opt_state
    )
    return kept_params, kept_state, loss, grad_norm, finite


def train_stage_i(example_name, seed, train_data, validation_data):
    spec = SDE_EXAMPLES[example_name]
    t_scale = float(spec["input_t_scale"])
    x_scale = float(spec["input_x_scale"])

    example_index = list(SDE_EXAMPLES).index(example_name)
    key = jax.random.PRNGKey(int(seed) + 10_000 + 100 * example_index)
    key_init, key_loop = jax.random.split(key)
    params = surrogate_model.init(key_init)
    opt_state = adam_init(params)

    best_params = tree_copy(params)
    best_val = float("inf")
    best_epoch = 0
    nonfinite_steps = 0
    history = []
    start = time.time()

    for epoch in range(1, int(CONFIG["BASELINE_EPOCHS"]) + 1):
        key_loop, key_batch = jax.random.split(key_loop)
        batch = sample_batch(key_batch, train_data, CONFIG["BASELINE_BATCH_SIZE"])
        params, opt_state, batch_loss, grad_norm, finite = baseline_train_step(
            params, opt_state, batch, t_scale, x_scale
        )
        if not bool(np.asarray(finite)):
            nonfinite_steps += 1

        should_eval = (
            epoch == 1
            or epoch % int(CONFIG["BASELINE_EVAL_EVERY"]) == 0
            or epoch == int(CONFIG["BASELINE_EPOCHS"])
        )
        if should_eval:
            train_nll = float(data_nll_loss(params, train_data, t_scale, x_scale))
            val_nll = float(data_nll_loss(params, validation_data, t_scale, x_scale))
            history.append({
                "epoch": epoch,
                "batch_nll": float(batch_loss),
                "train_nll": train_nll,
                "validation_nll": val_nll,
                "grad_norm": float(grad_norm),
            })
            if np.isfinite(val_nll) and val_nll < best_val:
                best_val = val_nll
                best_epoch = epoch
                best_params = tree_copy(params)

        if epoch == 1 or epoch % int(CONFIG["PRINT_EVERY"]) == 0:
            print(
                f"[{spec['label']} | Stage I] epoch {epoch:5d} | "
                f"batch={float(batch_loss): .5f} | best-val={best_val: .5f}"
            )

    if best_epoch == 0:
        raise RuntimeError(f"No finite Stage-I checkpoint for {example_name}.")

    return {
        "params": best_params,
        "best_epoch": best_epoch,
        "best_validation_nll": best_val,
        "runtime_sec": time.time() - start,
        "nonfinite_steps": nonfinite_steps,
        "history": history,
    }


## Cell 4 — Exact R=208 library, R=42 control, and mu-sweep libraries

In [ ]:
# ============================================================================
# Cell 4 — Exact R=208 library, R=42 control, and mu-sweep libraries
# ============================================================================


def _empty_record(name, family, source):
    return {
        "name": str(name),
        "family": str(family),
        "source": str(source),
        "tau_c0": 0.0,
        "tau_c1": 0.0,
        "tau_c2": 0.0,
        "tau_exp_amp": 0.0,
        "tau_exp_rate": 0.0,
        "xi_exp_rate": 0.0,
        "b": np.zeros(D, dtype=np.float32),
        "A": np.zeros((D, D), dtype=np.float32),
    }


def build_20d_library(mu_values, include_structured=True, include_time_distractors=True):
    """Construct the affine projectable library used by the spectral solver."""
    records = []

    def add(name, family, source, **kwargs):
        record = _empty_record(name, family, source)
        for key, value in kwargs.items():
            record[key] = value
        record["b"] = np.asarray(record["b"], dtype=np.float32)
        record["A"] = np.asarray(record["A"], dtype=np.float32)
        records.append(record)

    def e(i):
        v = np.zeros(D, dtype=np.float32)
        v[int(i)] = 1.0
        return v

    # Appendix C.3 order.
    add("time_translation__dt", "time_translation", "Appendix C.3", tau_c0=1.0)

    for i in range(D):
        add(
            f"translation__dx{i+1}",
            "coordinate_translation",
            "Appendix C.3",
            b=e(i),
        )

    add(
        "lifted_brownian_scaling__2t_dt_plus_x_dot_dx",
        "lifted_brownian_scaling",
        "Appendix C.3",
        tau_c1=2.0,
        A=np.eye(D, dtype=np.float32),
    )

    if include_time_distractors:
        add("time_scaling__t_dt", "time_distractor", "Appendix C.3", tau_c1=1.0)
        add("time_quadratic__t2_dt", "time_distractor", "Appendix C.3", tau_c2=1.0)

    for i in range(D):
        A = np.zeros((D, D), dtype=np.float32)
        A[i, i] = 1.0
        add(
            f"coordinate_scaling__x{i+1}_dx{i+1}",
            "coordinate_scaling",
            "Appendix C.3",
            A=A,
        )

    for mu in [float(v) for v in mu_values]:
        for i in range(D):
            add(
                f"exp_translation__mu_{mu:+.2f}__dx{i+1}",
                "exponential_translation",
                "Appendix C.3",
                xi_exp_rate=mu,
                b=e(i),
            )
        add(
            f"exp_projective__mu_{mu:+.2f}",
            "exponential_projective",
            "Appendix C.3",
            tau_exp_amp=1.0,
            tau_exp_rate=2.0 * mu,
            xi_exp_rate=2.0 * mu,
            A=mu * np.eye(D, dtype=np.float32),
        )

    if include_structured:
        for i in range(D):
            ip1 = (i + 1) % D
            b = e(i) - e(ip1)
            add(
                f"ring_difference__dx{i+1}_minus_dx{ip1+1}",
                "ring_difference",
                "Appendix C.3",
                b=b,
            )

        for i in range(D):
            ip1 = (i + 1) % D
            A = np.zeros((D, D), dtype=np.float32)
            A[i, ip1] = 1.0
            A[ip1, i] = -1.0
            add(
                f"ring_rotation__x{ip1+1}_dx{i+1}_minus_x{i+1}_dx{ip1+1}",
                "ring_rotation",
                "Appendix C.3",
                A=A,
            )

        for i in range(D):
            ip1 = (i + 1) % D
            im1 = (i - 1) % D

            A_plus = np.zeros((D, D), dtype=np.float32)
            A_plus[i, ip1] = 1.0
            add(
                f"ring_shear__x{ip1+1}_dx{i+1}",
                "ring_shear",
                "Appendix C.3",
                A=A_plus,
            )

            A_minus = np.zeros((D, D), dtype=np.float32)
            A_minus[i, im1] = 1.0
            add(
                f"ring_shear__x{im1+1}_dx{i+1}",
                "ring_shear",
                "Appendix C.3",
                A=A_minus,
            )

    arrays = (
        jnp.asarray([r["tau_c0"] for r in records], dtype=jnp.float32),
        jnp.asarray([r["tau_c1"] for r in records], dtype=jnp.float32),
        jnp.asarray([r["tau_c2"] for r in records], dtype=jnp.float32),
        jnp.asarray([r["tau_exp_amp"] for r in records], dtype=jnp.float32),
        jnp.asarray([r["tau_exp_rate"] for r in records], dtype=jnp.float32),
        jnp.asarray([r["xi_exp_rate"] for r in records], dtype=jnp.float32),
        jnp.asarray(np.stack([r["b"] for r in records]), dtype=jnp.float32),
        jnp.asarray(np.stack([r["A"] for r in records]), dtype=jnp.float32),
    )
    return {
        "records": records,
        "names": [r["name"] for r in records],
        "families": [r["family"] for r in records],
        "arrays": arrays,
        "size": len(records),
        "mu_values": tuple(float(v) for v in mu_values),
    }


def build_negative_control_library():
    """
    Deliberately mismatched R=42 control.

    It excludes every basic exact-symmetry family that caused the previous
    control to succeed spuriously: no time translations, coordinate
    translations, coordinate scalings, lifted Brownian scaling, ring
    differences, or nearest-neighbor ring generators.

    Contents:
      - 20 stride-7 long-range rotations;
      - 20 stride-7 one-way long-range shears;
      - 2 wrong-rate projective templates (mu = -3 and +3).
    """
    records = []

    def add(name, family, source, **kwargs):
        record = _empty_record(name, family, source)
        for key, value in kwargs.items():
            record[key] = value
        record["b"] = np.asarray(record["b"], dtype=np.float32)
        record["A"] = np.asarray(record["A"], dtype=np.float32)
        records.append(record)

    stride = 7

    for i in range(D):
        j = (i + stride) % D
        A = np.zeros((D, D), dtype=np.float32)
        A[i, j] = 1.0
        A[j, i] = -1.0
        add(
            f"control_longrange_rotation__x{j+1}_dx{i+1}_minus_x{i+1}_dx{j+1}",
            "control_longrange_rotation",
            "E6 deliberately mismatched control",
            A=A,
        )

    for i in range(D):
        j = (i + stride) % D
        A = np.zeros((D, D), dtype=np.float32)
        A[i, j] = 1.0
        add(
            f"control_longrange_shear__x{j+1}_dx{i+1}",
            "control_longrange_shear",
            "E6 deliberately mismatched control",
            A=A,
        )

    for mu in (-3.0, 3.0):
        add(
            f"control_wrong_rate_projective__mu_{mu:+.2f}",
            "control_wrong_rate_projective",
            "E6 deliberately mismatched control",
            tau_exp_amp=1.0,
            tau_exp_rate=2.0 * mu,
            xi_exp_rate=2.0 * mu,
            A=mu * np.eye(D, dtype=np.float32),
        )

    arrays = (
        jnp.asarray([r["tau_c0"] for r in records], dtype=jnp.float32),
        jnp.asarray([r["tau_c1"] for r in records], dtype=jnp.float32),
        jnp.asarray([r["tau_c2"] for r in records], dtype=jnp.float32),
        jnp.asarray([r["tau_exp_amp"] for r in records], dtype=jnp.float32),
        jnp.asarray([r["tau_exp_rate"] for r in records], dtype=jnp.float32),
        jnp.asarray([r["xi_exp_rate"] for r in records], dtype=jnp.float32),
        jnp.asarray(np.stack([r["b"] for r in records]), dtype=jnp.float32),
        jnp.asarray(np.stack([r["A"] for r in records]), dtype=jnp.float32),
    )
    return {
        "records": records,
        "names": [r["name"] for r in records],
        "families": [r["family"] for r in records],
        "arrays": arrays,
        "size": len(records),
        "mu_values": (-3.0, 3.0),
        "expected_rank": int(CONFIG["E6_EXPECTED_CONTROL_RANK"]),
        "library_role": "negative_control",
    }


# Exact manuscript library: 1+20+1+2+20+80+4+20+20+40 = 208.
COMPLETE_LIBRARY = build_20d_library(
    CONFIG["COMPLETE_MU_LIST"],
    include_structured=True,
    include_time_distractors=True,
)
COMPLETE_LIBRARY["expected_rank"] = int(CONFIG["E6_EXPECTED_COMPLETE_RANK"])
COMPLETE_LIBRARY["library_role"] = "complete"

# Deliberately mismatched R=42 negative control.
REDUCED_LIBRARY = build_negative_control_library()

# Complete-structure single-mu libraries: R = 124 non-exponential + 21 = 145.
MU_SWEEP_LIBRARIES = {
    float(mu): build_20d_library(
        [float(mu)],
        include_structured=True,
        include_time_distractors=True,
    )
    for mu in CONFIG["MU_SWEEP_LIST"]
}
for _mu, _library in MU_SWEEP_LIBRARIES.items():
    _library["expected_rank"] = int(CONFIG["E6_EXPECTED_SINGLE_MU_RANK"])
    _library["library_role"] = f"single_mu_{_mu:+.2f}"

if COMPLETE_LIBRARY["size"] != 208:
    raise RuntimeError(f"Complete library has R={COMPLETE_LIBRARY['size']}, expected 208.")
if REDUCED_LIBRARY["size"] != 42:
    raise RuntimeError(f"Reduced library has R={REDUCED_LIBRARY['size']}, expected 42.")
if any(lib["size"] != 145 for lib in MU_SWEEP_LIBRARIES.values()):
    raise RuntimeError("Every single-mu sweep library must contain 145 templates.")
if M_MAX > REDUCED_LIBRARY["size"]:
    raise RuntimeError("The negative-control library is too small for m_max.")


def _safe_exp(z):
    clip = float(CONFIG["BASIS_EXP_CLIP"])
    return jnp.exp(jnp.clip(z, -clip, clip))


def eval_all_basis_affine(arrays, t, x):
    (
        tau_c0,
        tau_c1,
        tau_c2,
        tau_exp_amp,
        tau_exp_rate,
        xi_exp_rate,
        basis_b,
        basis_A,
    ) = arrays

    tau_exp = _safe_exp(tau_exp_rate * t)
    xi_scale = _safe_exp(xi_exp_rate * t)
    tau = tau_c0 + tau_c1 * t + tau_c2 * t * t + tau_exp_amp * tau_exp
    tau_t = (
        tau_c1
        + 2.0 * tau_c2 * t
        + tau_exp_amp * tau_exp_rate * tau_exp
    )
    affine = basis_b + jnp.einsum("rij,j->ri", basis_A, x)
    xi = xi_scale[:, None] * affine
    xi_t = (xi_exp_rate * xi_scale)[:, None] * affine
    xi_x = xi_scale[:, None, None] * basis_A
    return tau, tau_t, xi, xi_t, xi_x


def eval_generators_from_W(W, arrays, t, x):
    (
        tau_c0,
        tau_c1,
        tau_c2,
        tau_exp_amp,
        tau_exp_rate,
        xi_exp_rate,
        basis_b,
        basis_A,
    ) = arrays

    tau_exp = _safe_exp(tau_exp_rate * t)
    xi_scale = _safe_exp(xi_exp_rate * t)
    tau_all = tau_c0 + tau_c1 * t + tau_c2 * t * t + tau_exp_amp * tau_exp
    tau_t_all = tau_c1 + 2.0 * tau_c2 * t + tau_exp_amp * tau_exp_rate * tau_exp

    weighted = W * xi_scale[None, :]
    weighted_t = W * (xi_exp_rate * xi_scale)[None, :]
    b_eff = jnp.einsum("mr,ri->mi", weighted, basis_b)
    A_eff = jnp.einsum("mr,rij->mij", weighted, basis_A)
    b_t_eff = jnp.einsum("mr,ri->mi", weighted_t, basis_b)
    A_t_eff = jnp.einsum("mr,rij->mij", weighted_t, basis_A)

    tau = W @ tau_all
    tau_t = W @ tau_t_all
    xi = b_eff + jnp.einsum("mij,j->mi", A_eff, x)
    xi_t = b_t_eff + jnp.einsum("mij,j->mi", A_t_eff, x)
    return tau, tau_t, xi, xi_t, A_eff


print("\n20D libraries")
print("  complete R              :", COMPLETE_LIBRARY["size"])
print("  negative-control R      :", REDUCED_LIBRARY["size"])
print("  each single-mu sweep R  :", next(iter(MU_SWEEP_LIBRARIES.values()))["size"])
print("  complete mu grid        :", COMPLETE_LIBRARY["mu_values"])
print("  mu sweep                :", list(MU_SWEEP_LIBRARIES))

## Cell 5 — normalized-rank spectral diagnostic and adaptive m<=15

In [ ]:
# ============================================================================
# Cell 5 — normalized-rank spectral diagnostic and adaptive m<=15
# ============================================================================


def _symmetrize(matrix):
    matrix = np.asarray(matrix, dtype=np.float64)
    return 0.5 * (matrix + matrix.T)


def solve_psd_generalized_eigen(H, G):
    """
    Solve H w = q G w after empirical unit-norm normalization of every
    generator template.

    If c denotes coefficients in normalized-template coordinates and
    s_r=sqrt(G_rr), then original-library coefficients are w_r=c_r/s_r.
    The returned vectors are already mapped back to those original
    coefficients, so they can be passed directly to Stage III.
    """
    H = _symmetrize(H)
    G = _symmetrize(G)
    R = int(G.shape[0])

    template_norms = np.sqrt(np.maximum(np.diag(G), 0.0))
    active = template_norms > float(CONFIG["E6_GRAM_ABS_TOL"])
    if int(np.sum(active)) < M_MAX:
        raise RuntimeError(
            f"Only {int(np.sum(active))} basis templates have nonzero empirical norm; "
            f"m_max={M_MAX}."
        )

    active_indices = np.where(active)[0]
    active_norms = template_norms[active]
    G_active = G[np.ix_(active_indices, active_indices)]
    G_unit = G_active / (active_norms[:, None] * active_norms[None, :])
    G_unit = _symmetrize(G_unit)

    # Eigenvalues of the normalized Gram matrix are squared singular values
    # of the stacked, unit-column-norm generator evaluation matrix.
    gvals_unit, U_unit = np.linalg.eigh(G_unit)
    gmax = max(float(np.max(gvals_unit)), 0.0)
    rank_threshold = max(
        float(CONFIG["E6_GRAM_ABS_TOL"]),
        float(CONFIG["E6_GRAM_REL_TOL"]) * gmax,
    )
    keep = gvals_unit > rank_threshold
    effective_rank = int(np.sum(keep))
    if effective_rank < M_MAX:
        raise RuntimeError(
            f"Normalized generator Gram rank {effective_rank} is below "
            f"m_max={M_MAX}."
        )

    # Whiten in normalized coordinates, then map back to coefficients of the
    # original unnormalized templates.
    whitening_unit = U_unit[:, keep] / np.sqrt(gvals_unit[keep])[None, :]
    whitening = np.zeros((R, effective_rank), dtype=np.float64)
    whitening[active_indices, :] = whitening_unit / active_norms[:, None]

    H_white = _symmetrize(whitening.T @ H @ whitening)
    H_white += float(CONFIG["E6_MATRIX_RIDGE"]) * np.eye(
        H_white.shape[0], dtype=np.float64
    )

    q, V = np.linalg.eigh(H_white)
    order = np.argsort(q)
    q = np.maximum(q[order], 0.0)
    vectors = whitening @ V[:, order]

    # Explicit G-normalization removes accumulated roundoff.
    for k in range(vectors.shape[1]):
        denom = float(vectors[:, k].T @ G @ vectors[:, k])
        if denom > 0.0:
            vectors[:, k] /= math.sqrt(denom)

    return {
        "scores": q,
        "vectors": vectors,
        "r_effective": effective_rank,
        "whitening": whitening,
        "template_norms": template_norms,
        "active_template_mask": active,
        "normalized_gram_eigenvalues": gvals_unit,
        "rank_threshold": rank_threshold,
    }


def rayleigh_scores(H, G, W_columns):
    numerator = np.einsum("ik,ij,jk->k", W_columns, H, W_columns)
    denominator = np.einsum("ik,ij,jk->k", W_columns, G, W_columns)
    return np.maximum(numerator / np.maximum(denominator, 1.0e-30), 0.0)


def cumulative_means(values):
    values = np.asarray(values, dtype=np.float64)
    return np.cumsum(values) / np.arange(1, values.size + 1, dtype=np.float64)


def g_orthonormalize(W_columns, G):
    gram = _symmetrize(W_columns.T @ G @ W_columns)
    vals, vecs = np.linalg.eigh(gram)
    vals = np.maximum(vals, 1.0e-12)
    inv_sqrt = vecs @ np.diag(1.0 / np.sqrt(vals)) @ vecs.T
    return W_columns @ inv_sqrt


def principal_angles_deg(Wa, Wb, G):
    Qa = g_orthonormalize(Wa, G)
    Qb = g_orthonormalize(Wb, G)
    singular = np.linalg.svd(Qa.T @ G @ Qb, compute_uv=False)
    singular = np.clip(singular, 0.0, 1.0)
    return np.degrees(np.arccos(singular))


def random_subspace_score_table(H, G, solution, rng):
    """
    Return shape (n_random, M_MAX). Column k-1 contains the mean Rayleigh
    score of a random k-dimensional subspace.
    """
    whitening = solution["whitening"]
    k_eff = int(whitening.shape[1])
    table = np.zeros(
        (int(CONFIG["E6_N_RANDOM_SUBSPACES"]), M_MAX), dtype=np.float64
    )

    for draw in range(table.shape[0]):
        Z = rng.normal(size=(k_eff, M_MAX))
        Qrand, _ = np.linalg.qr(Z, mode="reduced")
        W = whitening @ Qrand[:, :M_MAX]
        mode_scores = rayleigh_scores(H, G, W)
        table[draw, :] = cumulative_means(mode_scores)

    return table


def bootstrap_subspace_angle_table(H_points, G_points, base_W_max, rng):
    """
    Rediscover the first M_MAX modes on five bootstrap samples. For each
    k=1,...,M_MAX, record the largest and mean principal angle between the
    base and bootstrap k-dimensional subspaces.
    """
    n_points = int(H_points.shape[0])
    G_ref = _symmetrize(np.mean(G_points, axis=0))
    n_boot = int(CONFIG["E6_BOOTSTRAP_REPEATS"])
    max_angle_table = np.full((n_boot, M_MAX), 90.0, dtype=np.float64)
    mean_angle_table = np.full((n_boot, M_MAX), 90.0, dtype=np.float64)

    for boot in range(n_boot):
        idx = rng.integers(0, n_points, size=n_points)
        H_b = _symmetrize(np.mean(H_points[idx], axis=0))
        G_b = _symmetrize(np.mean(G_points[idx], axis=0))
        try:
            solution = solve_psd_generalized_eigen(H_b, G_b)
            W_b_max = solution["vectors"][:, :M_MAX]
            for k in range(1, M_MAX + 1):
                angles = principal_angles_deg(
                    base_W_max[:, :k], W_b_max[:, :k], G_ref
                )
                max_angle_table[boot, k - 1] = float(np.max(angles))
                mean_angle_table[boot, k - 1] = float(np.mean(angles))
        except (np.linalg.LinAlgError, RuntimeError, FloatingPointError):
            pass

    return {
        "max_angle_table_deg": max_angle_table,
        "mean_angle_table_deg": mean_angle_table,
        "bootstrap_max_angle_by_m_deg": np.mean(max_angle_table, axis=0),
        "bootstrap_mean_angle_by_m_deg": np.mean(mean_angle_table, axis=0),
    }


def make_operator_functions(spec, library):
    drift_fn = spec["drift_fn"]
    sigma_fn = spec["sigma_fn"]
    arrays = library["arrays"]
    R = int(library["size"])
    t_scale = float(spec["input_t_scale"])
    x_scale = float(spec["input_x_scale"])

    def coefficient_values_and_jacobian_surrogate(params, z):
        def concat_coeff(zz):
            f, sigma = surrogate_f_sigma(params, zz[0], zz[1:], t_scale, x_scale)
            return jnp.concatenate([f, sigma])

        values = concat_coeff(z)
        jacobian = jax.jacfwd(concat_coeff)(z)
        return values[:D], values[D:], jacobian[:D], jacobian[D:]

    def coefficient_values_and_jacobian_truth(z):
        def concat_coeff(zz):
            return jnp.concatenate([
                drift_fn(zz[0], zz[1:]),
                sigma_fn(zz[0], zz[1:]),
            ])

        values = concat_coeff(z)
        jacobian = jax.jacfwd(concat_coeff)(z)
        return values[:D], values[D:], jacobian[:D], jacobian[D:]

    def basis_residuals(f, sigma, J_f, J_sigma, z):
        f_t, f_x = J_f[:, 0], J_f[:, 1:]
        sigma_t, sigma_x = J_sigma[:, 0], J_sigma[:, 1:]
        tau, tau_t, xi, xi_t, xi_x = eval_all_basis_affine(
            arrays, z[0], z[1:]
        )

        R_f = (
            xi_t
            + jnp.einsum("j,rij->ri", f, xi_x)
            - jnp.einsum("rj,ij->ri", xi, f_x)
            - tau[:, None] * f_t[None, :]
            - tau_t[:, None] * f[None, :]
        )

        first_sigma = xi_x * sigma[None, None, :]
        correction = (
            jnp.einsum("rj,ij->ri", xi, sigma_x)
            + tau[:, None] * sigma_t[None, :]
            + 0.5 * tau_t[:, None] * sigma[None, :]
        )
        eye = jnp.eye(D, dtype=jnp.float32)
        R_sigma = first_sigma - correction[:, :, None] * eye[None, :, :]

        if CONFIG["E6_NORMALIZE_OPERATOR_RESIDUAL"]:
            f_scale = 1.0 + jnp.abs(f) + jnp.abs(f_t) + jnp.sqrt(
                jnp.sum(f_x**2, axis=1) + 1.0e-12
            )
            sigma_scale = 1.0 + jnp.abs(sigma) + jnp.abs(sigma_t) + jnp.sqrt(
                jnp.sum(sigma_x**2, axis=1) + 1.0e-12
            )
            R_f = R_f / f_scale[None, :]
            pair_scale = jnp.sqrt(
                sigma_scale[:, None] * sigma_scale[None, :]
            )
            R_sigma = R_sigma / pair_scale[None, :, :]

        return R_f, R_sigma, tau, xi

    def contribution_surrogate(params, z):
        f, sigma, J_f, J_sigma = coefficient_values_and_jacobian_surrogate(
            params, z
        )
        R_f, R_sigma, tau, xi = basis_residuals(
            f, sigma, J_f, J_sigma, z
        )
        flat_sigma = R_sigma.reshape((R, -1))
        H = (
            (R_f @ R_f.T) / float(D)
            + (flat_sigma @ flat_sigma.T) / float(D * D)
        )
        G = (tau[:, None] * tau[None, :] + xi @ xi.T) / float(Q)
        return H, G

    def contribution_truth(z):
        f, sigma, J_f, J_sigma = coefficient_values_and_jacobian_truth(z)
        R_f, R_sigma, tau, xi = basis_residuals(
            f, sigma, J_f, J_sigma, z
        )
        flat_sigma = R_sigma.reshape((R, -1))
        H = (
            (R_f @ R_f.T) / float(D)
            + (flat_sigma @ flat_sigma.T) / float(D * D)
        )
        G = (tau[:, None] * tau[None, :] + xi @ xi.T) / float(Q)
        return H, G

    return {
        "surrogate_batch": jax.jit(
            jax.vmap(contribution_surrogate, in_axes=(None, 0))
        ),
        "truth_batch": jax.jit(jax.vmap(contribution_truth)),
    }


def compute_pointwise_operator_matrices(kind, params, tx, functions):
    tx = jnp.asarray(tx, dtype=jnp.float32)
    chunk = int(CONFIG["E6_OPERATOR_CHUNK_SIZE"])
    H_parts, G_parts = [], []

    for start in range(0, int(tx.shape[0]), chunk):
        tx_chunk = tx[start:start + chunk]
        if kind == "surrogate":
            H, G = functions["surrogate_batch"](params, tx_chunk)
        elif kind == "truth":
            H, G = functions["truth_batch"](tx_chunk)
        else:
            raise ValueError(f"Unknown operator kind {kind}.")
        H_parts.append(np.asarray(H, dtype=np.float64))
        G_parts.append(np.asarray(G, dtype=np.float64))

    return np.concatenate(H_parts, axis=0), np.concatenate(G_parts, axis=0)


def diagnose_library(kind, params, spec, library, tx_train, tx_validation, rng_seed):
    functions = make_operator_functions(spec, library)
    Htr_pts, Gtr_pts = compute_pointwise_operator_matrices(
        kind, params, tx_train, functions
    )
    Hva_pts, Gva_pts = compute_pointwise_operator_matrices(
        kind, params, tx_validation, functions
    )

    Htr = _symmetrize(np.mean(Htr_pts, axis=0))
    Gtr = _symmetrize(np.mean(Gtr_pts, axis=0))
    Hva = _symmetrize(np.mean(Hva_pts, axis=0))
    Gva = _symmetrize(np.mean(Gva_pts, axis=0))

    solution = solve_psd_generalized_eigen(Htr, Gtr)
    expected_rank = int(library["expected_rank"])
    rank_verified = int(solution["r_effective"]) == expected_rank
    if (
        bool(CONFIG["E6_ENFORCE_LIBRARY_RANK"])
        and not bool(CONFIG["DEBUG_FAST"])
        and not rank_verified
    ):
        raise RuntimeError(
            f"{library['library_role']} normalized Gram rank is "
            f"{solution['r_effective']}, expected {expected_rank}. "
            "Do not interpret E6 until this rank check passes."
        )

    W_max_columns = solution["vectors"][:, :M_MAX]
    train_scores = solution["scores"][:M_MAX]
    validation_scores = rayleigh_scores(Hva, Gva, W_max_columns)

    train_aggregate_by_m = cumulative_means(train_scores)
    validation_aggregate_by_m = cumulative_means(validation_scores)
    inflation_by_m = validation_aggregate_by_m / np.maximum(
        train_aggregate_by_m, 1.0e-30
    )

    rng = np.random.default_rng(int(rng_seed))
    random_table = random_subspace_score_table(Hva, Gva, solution, rng)
    null_threshold_by_m = np.quantile(
        random_table,
        float(CONFIG["E6_RANDOM_NULL_QUANTILE"]),
        axis=0,
    )
    null_median_by_m = np.median(random_table, axis=0)
    score_to_null_by_m = validation_aggregate_by_m / np.maximum(
        null_threshold_by_m, 1.0e-30
    )
    score_to_median_by_m = validation_aggregate_by_m / np.maximum(
        null_median_by_m, 1.0e-30
    )

    bootstrap = bootstrap_subspace_angle_table(
        Htr_pts, Gtr_pts, W_max_columns, rng
    )
    boot_max_by_m = bootstrap["bootstrap_max_angle_by_m_deg"]
    boot_mean_by_m = bootstrap["bootstrap_mean_angle_by_m_deg"]

    numerical_valid = bool(
        np.all(np.isfinite(train_scores))
        and np.all(np.isfinite(validation_scores))
        and np.all(np.isfinite(W_max_columns))
        and solution["r_effective"] >= M_MAX
        and rank_verified
    )

    gate_tests_by_m = []
    accepted_dimensions = []
    for k in range(1, M_MAX + 1):
        i = k - 1
        tests = {
            "numerical_valid": numerical_valid,
            "absolute_score": validation_aggregate_by_m[i]
                <= float(CONFIG["E6_GATE_MAX_VALIDATION_SCORE"]),
            "random_null": score_to_null_by_m[i]
                <= float(CONFIG["E6_GATE_MAX_SCORE_TO_NULL"]),
            "median_separation": score_to_median_by_m[i]
                <= float(CONFIG["E6_GATE_MAX_SCORE_TO_MEDIAN"]),
            "train_validation": inflation_by_m[i]
                <= float(CONFIG["E6_GATE_MAX_TRAIN_VAL_INFLATION"]),
            "bootstrap_stability": boot_max_by_m[i]
                <= float(CONFIG["E6_GATE_MAX_BOOTSTRAP_MAX_ANGLE_DEG"]),
        }
        accepted = all(tests.values())
        gate_tests_by_m.append(tests)
        if accepted:
            accepted_dimensions.append(k)

    selected_m = max(accepted_dimensions) if accepted_dimensions else 0
    decision = "ACCEPT" if selected_m > 0 else "ABSTAIN"

    if selected_m > 0:
        selected_index = selected_m - 1
        reason = f"accepted largest stable near-null prefix m={selected_m}"
        selected_W = W_max_columns[:, :selected_m].T
        selected_tests = gate_tests_by_m[selected_index]
    else:
        selected_index = 0
        failed = [
            name for name, passed in gate_tests_by_m[0].items() if not passed
        ]
        reason = (
            "no dimension in 1..m_max passed; m=1 failed: "
            + ", ".join(failed)
        )
        selected_W = np.zeros(
            (0, int(library["size"])), dtype=np.float64
        )
        selected_tests = gate_tests_by_m[0]

    result = {
        "r_raw": int(library["size"]),
        "r_effective": int(solution["r_effective"]),
        "expected_rank": expected_rank,
        "rank_verified": bool(rank_verified),
        "m_max": M_MAX,
        "selected_m": int(selected_m),
        "train_scores": np.asarray(train_scores, dtype=np.float64),
        # These are held-out Rayleigh scores of train-discovered modes; they
        # are not separately sorted validation eigenvalues.
        "validation_scores": np.asarray(validation_scores, dtype=np.float64),
        "validation_scores_sorted": np.sort(
            np.asarray(validation_scores, dtype=np.float64)
        ),
        "train_aggregate_by_m": np.asarray(
            train_aggregate_by_m, dtype=np.float64
        ),
        "validation_aggregate_by_m": np.asarray(
            validation_aggregate_by_m, dtype=np.float64
        ),
        # Keep this fixed-m_max score for fair complete/control comparison.
        "train_aggregate_score": float(train_aggregate_by_m[-1]),
        "validation_aggregate_score": float(
            validation_aggregate_by_m[-1]
        ),
        "selected_train_aggregate_score": (
            float(train_aggregate_by_m[selected_index])
            if selected_m > 0 else float("nan")
        ),
        "selected_validation_aggregate_score": (
            float(validation_aggregate_by_m[selected_index])
            if selected_m > 0 else float("nan")
        ),
        "validation_singular_values": np.sqrt(
            np.maximum(validation_scores, 0.0)
        ),
        # W is the full 15-direction discovery used only by ungated controls.
        "W": np.asarray(W_max_columns.T, dtype=np.float64),
        # W_selected is the adaptive Stage-III subspace.
        "W_selected": np.asarray(selected_W, dtype=np.float64),
        "null_threshold_by_m": np.asarray(
            null_threshold_by_m, dtype=np.float64
        ),
        "null_median_by_m": np.asarray(
            null_median_by_m, dtype=np.float64
        ),
        "score_to_null_by_m": np.asarray(
            score_to_null_by_m, dtype=np.float64
        ),
        "score_to_median_by_m": np.asarray(
            score_to_median_by_m, dtype=np.float64
        ),
        "train_to_validation_inflation_by_m": np.asarray(
            inflation_by_m, dtype=np.float64
        ),
        "bootstrap_max_angle_by_m_deg": np.asarray(
            boot_max_by_m, dtype=np.float64
        ),
        "bootstrap_mean_angle_by_m_deg": np.asarray(
            boot_mean_by_m, dtype=np.float64
        ),
        # Scalar aliases refer to the selected dimension, or m=1 on abstention.
        "null_threshold": float(null_threshold_by_m[selected_index]),
        "null_median": float(null_median_by_m[selected_index]),
        "score_to_null": float(score_to_null_by_m[selected_index]),
        "score_to_median": float(score_to_median_by_m[selected_index]),
        "train_to_validation_inflation": float(
            inflation_by_m[selected_index]
        ),
        "bootstrap_max_angle_deg": float(boot_max_by_m[selected_index]),
        "bootstrap_mean_angle_deg": float(boot_mean_by_m[selected_index]),
        "bootstrap_max_angles_deg": np.asarray(
            bootstrap["max_angle_table_deg"][:, selected_index],
            dtype=np.float64,
        ),
        "gate_tests": selected_tests,
        "gate_tests_by_m": gate_tests_by_m,
        "decision": decision,
        "reason": reason,
    }

    del functions, Htr_pts, Gtr_pts, Hva_pts, Gva_pts
    gc.collect()
    return result


def generator_residuals_point(params, W, arrays, z, t_scale, x_scale):
    def concat_coeff(zz):
        f, sigma = surrogate_f_sigma(
            params, zz[0], zz[1:], t_scale, x_scale
        )
        return jnp.concatenate([f, sigma])

    values = concat_coeff(z)
    jacobian = jax.jacfwd(concat_coeff)(z)
    f, sigma = values[:D], values[D:]
    J_f, J_sigma = jacobian[:D], jacobian[D:]
    f_t, f_x = J_f[:, 0], J_f[:, 1:]
    sigma_t, sigma_x = J_sigma[:, 0], J_sigma[:, 1:]

    tau, tau_t, xi, xi_t, xi_x = eval_generators_from_W(
        W, arrays, z[0], z[1:]
    )
    R_f = (
        xi_t
        + jnp.einsum("j,mij->mi", f, xi_x)
        - jnp.einsum("mj,ij->mi", xi, f_x)
        - tau[:, None] * f_t[None, :]
        - tau_t[:, None] * f[None, :]
    )

    first_sigma = xi_x * sigma[None, None, :]
    correction = (
        jnp.einsum("mj,ij->mi", xi, sigma_x)
        + tau[:, None] * sigma_t[None, :]
        + 0.5 * tau_t[:, None] * sigma[None, :]
    )
    R_sigma = (
        first_sigma
        - correction[:, :, None]
        * jnp.eye(D, dtype=jnp.float32)[None, :, :]
    )
    return R_f, R_sigma


def multi_generator_determining_loss(params, W, arrays, tx, t_scale, x_scale):
    if int(W.shape[0]) == 0:
        return jnp.asarray(0.0, dtype=jnp.float32)
    R_f, R_sigma = jax.vmap(
        lambda z: generator_residuals_point(
            params, W, arrays, z, t_scale, x_scale
        )
    )(tx)
    return jnp.mean(R_f**2) + jnp.mean(R_sigma**2)


## Cell 6 — One-seed, three-example E6 execution

In [ ]:
# ============================================================================
# Cell 6 — One-seed, three-example E6 execution
# ============================================================================


def stage_f_objective(params, W, arrays, batch, tx_extra, t_scale, x_scale):
    data_loss = data_nll_loss(params, batch, t_scale, x_scale)
    t_batch, x_batch, _, _ = batch
    n_data = min(int(CONFIG["STAGE_F_N_DATA_SYM_POINTS"]), int(t_batch.shape[0]))
    tx_data = jnp.concatenate([t_batch[:n_data, None], x_batch[:n_data]], axis=1)
    tx_sym = jnp.concatenate([tx_data, tx_extra], axis=0)
    sym_loss = multi_generator_determining_loss(
        params, W, arrays, tx_sym, t_scale, x_scale
    )
    total = data_loss + float(CONFIG["STAGE_F_LAMBDA_SYM_DET"]) * sym_loss
    return total, {"total": total, "data": data_loss, "sym_det": sym_loss}


@jax.jit
def stage_f_train_step(params, opt_state, W, arrays, batch, tx_extra, t_scale, x_scale):
    (loss, aux), grads = jax.value_and_grad(stage_f_objective, has_aux=True)(
        params, W, arrays, batch, tx_extra, t_scale, x_scale
    )
    grads, grad_norm = clip_grad_tree(grads, CONFIG["GLOBAL_GRAD_CLIP"])
    new_params, new_state = adam_update(params, grads, opt_state, CONFIG["STAGE_F_LR"])
    finite = jnp.logical_and(jnp.isfinite(loss), tree_all_finite(new_params))
    kept_params = jax.tree_util.tree_map(
        lambda new, old: jnp.where(finite, new, old), new_params, params
    )
    kept_state = jax.tree_util.tree_map(
        lambda new, old: jnp.where(finite, new, old), new_state, opt_state
    )
    return kept_params, kept_state, aux, grad_norm, finite


@jax.jit
def data_only_train_step(params, opt_state, batch, t_scale, x_scale):
    loss, grads = jax.value_and_grad(data_nll_loss)(
        params, batch, t_scale, x_scale
    )
    grads, grad_norm = clip_grad_tree(grads, CONFIG["GLOBAL_GRAD_CLIP"])
    new_params, new_state = adam_update(params, grads, opt_state, CONFIG["STAGE_F_LR"])
    finite = jnp.logical_and(jnp.isfinite(loss), tree_all_finite(new_params))
    kept_params = jax.tree_util.tree_map(
        lambda new, old: jnp.where(finite, new, old), new_params, params
    )
    kept_state = jax.tree_util.tree_map(
        lambda new, old: jnp.where(finite, new, old), new_state, opt_state
    )
    return kept_params, kept_state, loss, grad_norm, finite


def train_stage_f(example_name, seed, baseline_params, W_np, library, train_data, validation_data, label):
    spec = SDE_EXAMPLES[example_name]
    t_scale = float(spec["input_t_scale"])
    x_scale = float(spec["input_x_scale"])
    W = jnp.asarray(W_np, dtype=jnp.float32)
    arrays = library["arrays"]

    params = tree_copy(baseline_params)
    opt_state = adam_init(params)
    best_params = tree_copy(params)
    best_val = float("inf")
    best_epoch = 0
    nonfinite_steps = 0
    history = []

    example_index = list(SDE_EXAMPLES).index(example_name)
    key = jax.random.PRNGKey(int(seed) + 30_000 + 100 * example_index)
    start = time.time()

    for epoch in range(1, int(CONFIG["STAGE_F_EPOCHS"]) + 1):
        key, key_batch, key_extra = jax.random.split(key, 3)
        batch = sample_batch(key_batch, train_data, CONFIG["STAGE_F_BATCH_SIZE"])
        tx_extra = sample_tx_box(key_extra, CONFIG["STAGE_F_N_EXTRA_SYM_POINTS"], spec)
        params, opt_state, aux, grad_norm, finite = stage_f_train_step(
            params, opt_state, W, arrays, batch, tx_extra, t_scale, x_scale
        )
        if not bool(np.asarray(finite)):
            nonfinite_steps += 1

        should_eval = (
            epoch == 1
            or epoch % int(CONFIG["STAGE_F_EVAL_EVERY"]) == 0
            or epoch == int(CONFIG["STAGE_F_EPOCHS"])
        )
        if should_eval:
            val_nll = float(data_nll_loss(params, validation_data, t_scale, x_scale))
            history.append({
                "epoch": epoch,
                "validation_nll": val_nll,
                "total": float(aux["total"]),
                "data": float(aux["data"]),
                "sym_det": float(aux["sym_det"]),
                "grad_norm": float(grad_norm),
            })
            if np.isfinite(val_nll) and val_nll < best_val:
                best_val = val_nll
                best_epoch = epoch
                best_params = tree_copy(params)

        if epoch == 1 or epoch % int(CONFIG["PRINT_EVERY"]) == 0:
            print(
                f"[{spec['label']} | {label}] epoch {epoch:5d} | "
                f"data={float(aux['data']): .5f} | sym={float(aux['sym_det']):.4e} | "
                f"best-val={best_val: .5f}"
            )

    if best_epoch == 0:
        raise RuntimeError(f"No finite Stage-III checkpoint for {label}.")
    return {
        "params": best_params,
        "best_epoch": best_epoch,
        "best_validation_nll": best_val,
        "runtime_sec": time.time() - start,
        "nonfinite_steps": nonfinite_steps,
        "history": history,
    }


def train_data_only_continuation(example_name, seed, baseline_params, train_data, validation_data):
    spec = SDE_EXAMPLES[example_name]
    t_scale = float(spec["input_t_scale"])
    x_scale = float(spec["input_x_scale"])
    params = tree_copy(baseline_params)
    opt_state = adam_init(params)
    best_params = tree_copy(params)
    best_val = float(data_nll_loss(params, validation_data, t_scale, x_scale))
    best_epoch = 0
    nonfinite_steps = 0

    example_index = list(SDE_EXAMPLES).index(example_name)
    key = jax.random.PRNGKey(int(seed) + 30_000 + 100 * example_index)
    start = time.time()

    for epoch in range(1, int(CONFIG["STAGE_F_EPOCHS"]) + 1):
        key, key_batch, _ = jax.random.split(key, 3)
        batch = sample_batch(key_batch, train_data, CONFIG["STAGE_F_BATCH_SIZE"])
        params, opt_state, loss, grad_norm, finite = data_only_train_step(
            params, opt_state, batch, t_scale, x_scale
        )
        if not bool(np.asarray(finite)):
            nonfinite_steps += 1

        if (
            epoch == 1
            or epoch % int(CONFIG["STAGE_F_EVAL_EVERY"]) == 0
            or epoch == int(CONFIG["STAGE_F_EPOCHS"])
        ):
            val_nll = float(data_nll_loss(params, validation_data, t_scale, x_scale))
            if np.isfinite(val_nll) and val_nll < best_val:
                best_val = val_nll
                best_epoch = epoch
                best_params = tree_copy(params)

        if epoch == 1 or epoch % int(CONFIG["PRINT_EVERY"]) == 0:
            print(
                f"[{spec['label']} | data-only continuation] epoch {epoch:5d} | "
                f"batch={float(loss): .5f} | best-val={best_val: .5f}"
            )

    return {
        "params": best_params,
        "best_epoch": best_epoch,
        "best_validation_nll": best_val,
        "runtime_sec": time.time() - start,
        "nonfinite_steps": nonfinite_steps,
    }


def coefficient_mse_on_tx(params, tx, spec):
    t_scale = float(spec["input_t_scale"])
    x_scale = float(spec["input_x_scale"])
    pred_f, pred_sigma = jax.vmap(
        lambda z: surrogate_f_sigma(params, z[0], z[1:], t_scale, x_scale)
    )(tx)
    truth_f = jax.vmap(lambda z: spec["drift_fn"](z[0], z[1:]))(tx)
    truth_sigma = jax.vmap(lambda z: spec["sigma_fn"](z[0], z[1:]))(tx)
    return (
        float(jnp.mean((pred_f - truth_f) ** 2)),
        float(jnp.mean((pred_sigma - truth_sigma) ** 2)),
    )


def make_branch_record(
    branch_name,
    params,
    baseline_params,
    example_name,
    train_data,
    validation_data,
    test_data,
    tx_eval,
    tx_residual,
    W_np,
    library,
    runtime_sec,
    nonfinite_steps,
    effective_m,
    lambda_gl,
    abstained,
):
    spec = SDE_EXAMPLES[example_name]
    t_scale = float(spec["input_t_scale"])
    x_scale = float(spec["input_x_scale"])
    W = jnp.asarray(W_np, dtype=jnp.float32)
    arrays = library["arrays"]
    drift_mse, sigma_mse = coefficient_mse_on_tx(params, tx_eval, spec)
    residual_before = float(multi_generator_determining_loss(
        baseline_params, W, arrays, tx_residual, t_scale, x_scale
    ))
    residual_after = float(multi_generator_determining_loss(
        params, W, arrays, tx_residual, t_scale, x_scale
    ))
    return {
        "branch_name": branch_name,
        "effective_m": int(effective_m),
        "lambda_gl": float(lambda_gl),
        "train_nll": float(data_nll_loss(params, train_data, t_scale, x_scale)),
        "validation_nll": float(data_nll_loss(params, validation_data, t_scale, x_scale)),
        "test_nll": float(data_nll_loss(params, test_data, t_scale, x_scale)),
        "drift_mse": drift_mse,
        "sigma_mse": sigma_mse,
        "det_residual_before": residual_before,
        "det_residual_after": residual_after,
        "runtime_sec": float(runtime_sec),
        "nonfinite_steps": int(nonfinite_steps),
        "abstained": bool(abstained),
        "param_max_abs_diff_vs_baseline": tree_max_abs_diff(params, baseline_params),
    }


def make_data_only_record(
    fit,
    baseline_params,
    example_name,
    train_data,
    validation_data,
    test_data,
    tx_eval,
    tx_residual,
    full_W,
):
    return make_branch_record(
        "data_only_continuation",
        fit["params"],
        baseline_params,
        example_name,
        train_data,
        validation_data,
        test_data,
        tx_eval,
        tx_residual,
        full_W,
        COMPLETE_LIBRARY,
        fit["runtime_sec"],
        fit["nonfinite_steps"],
        0,
        0.0,
        False,
    )


def make_exact_abstention_alias(vanilla_record, branch_name):
    record = copy.deepcopy(vanilla_record)
    record.update({
        "branch_name": branch_name,
        "effective_m": 0,
        "lambda_gl": 0.0,
        "abstained": True,
        "param_max_abs_diff_vs_baseline": 0.0,
    })
    return record


def run_one_example(example_name, seed):
    spec = SDE_EXAMPLES[example_name]
    print("\n" + "=" * 120)
    print(f"E6 — {spec['label']} — seed {seed}")
    print("=" * 120)

    data = make_example_data(example_name, seed)
    train_data = data["train"]
    validation_data = data["validation"]
    test_data = data["test"]
    print(
        "trajectory split:",
        int(data["train_paths"].shape[0]),
        int(data["validation_paths"].shape[0]),
        int(data["test_paths"].shape[0]),
    )
    print(
        "increment counts:",
        int(train_data[0].shape[0]),
        int(validation_data[0].shape[0]),
        int(test_data[0].shape[0]),
    )

    stage_i = train_stage_i(example_name, seed, train_data, validation_data)
    baseline_params = stage_i["params"]

    example_index = list(SDE_EXAMPLES).index(example_name)
    key = jax.random.PRNGKey(int(seed) + 40_000 + 100 * example_index)
    keys = jax.random.split(key, 7)
    tx_spec_train = sample_tx_box(keys[0], CONFIG["E6_SPECTRAL_TRAIN_POINTS"], spec)
    tx_spec_val = sample_tx_box(keys[1], CONFIG["E6_SPECTRAL_VALIDATION_POINTS"], spec)
    tx_oracle_train = sample_tx_box(keys[2], CONFIG["E6_ORACLE_POINTS"], spec)
    tx_oracle_val = sample_tx_box(keys[3], CONFIG["E6_ORACLE_POINTS"], spec)
    tx_eval = sample_tx_box(keys[4], CONFIG["N_EVAL_POINTS"], spec)
    tx_residual = sample_tx_box(keys[5], CONFIG["N_RESIDUAL_EVAL_POINTS"], spec)

    print("[oracle] complete library")
    oracle_full = diagnose_library(
        "truth", None, spec, COMPLETE_LIBRARY, tx_oracle_train, tx_oracle_val,
        rng_seed=seed + 101 + 1000 * example_index,
    )
    print("[oracle] reduced library")
    oracle_reduced = diagnose_library(
        "truth", None, spec, REDUCED_LIBRARY, tx_oracle_train, tx_oracle_val,
        rng_seed=seed + 102 + 1000 * example_index,
    )

    print("[surrogate] complete library")
    surrogate_full = diagnose_library(
        "surrogate", baseline_params, spec, COMPLETE_LIBRARY, tx_spec_train, tx_spec_val,
        rng_seed=seed + 201 + 1000 * example_index,
    )
    print("[surrogate] reduced library")
    surrogate_reduced = diagnose_library(
        "surrogate", baseline_params, spec, REDUCED_LIBRARY, tx_spec_train, tx_spec_val,
        rng_seed=seed + 202 + 1000 * example_index,
    )

    print("\n[mu sweep] frozen Stage-I surrogate")
    mu_sweep = {}
    for mu_index, mu in enumerate(CONFIG["MU_SWEEP_LIST"]):
        mu = float(mu)
        diagnostic = diagnose_library(
            "surrogate",
            baseline_params,
            spec,
            MU_SWEEP_LIBRARIES[mu],
            tx_spec_train,
            tx_spec_val,
            rng_seed=seed + 300 + 17 * mu_index + 1000 * example_index,
        )
        mu_sweep[mu] = diagnostic
        selected_score = diagnostic["selected_validation_aggregate_score"]
        selected_score_text = (
            f"{selected_score:.4e}"
            if np.isfinite(selected_score)
            else "n/a"
        )
        print(
            f"  mu={mu:+.2f} | m*={diagnostic['selected_m']:2d} | "
            f"selected-score={selected_score_text} | "
            f"m_max-score={diagnostic['validation_aggregate_score']:.4e} | "
            f"angle={diagnostic['bootstrap_max_angle_deg']:.2f} | "
            f"{diagnostic['decision']}"
        )

    accepted_mu = [
        mu for mu, diag in mu_sweep.items() if diag["selected_m"] > 0
    ]
    if accepted_mu:
        selected_mu = min(
            accepted_mu,
            key=lambda mu: mu_sweep[mu][
                "selected_validation_aggregate_score"
            ],
        )
    else:
        # Still report the best fixed-m_max spectrum, but the downstream
        # mu-selected branch will abstain because no candidate passed.
        selected_mu = min(
            mu_sweep,
            key=lambda mu: mu_sweep[mu]["validation_aggregate_score"],
        )
    print(
        f"Validation-selected mu: {selected_mu:+.2f} | "
        f"selected m={mu_sweep[selected_mu]['selected_m']}"
    )

    full_W_max = surrogate_full["W"]
    reduced_W_max = surrogate_reduced["W"]
    full_W_selected = surrogate_full["W_selected"]
    reduced_W_selected = surrogate_reduced["W_selected"]
    selected_mu_W = mu_sweep[selected_mu]["W_selected"]

    # Vanilla uses the 15-direction complete-library discovery only for
    # residual reporting; it receives no symmetry update.
    vanilla = make_branch_record(
        "vanilla",
        baseline_params,
        baseline_params,
        example_name,
        train_data,
        validation_data,
        test_data,
        tx_eval,
        tx_residual,
        full_W_max,
        COMPLETE_LIBRARY,
        stage_i["runtime_sec"],
        stage_i["nonfinite_steps"],
        0,
        0.0,
        False,
    )

    print("\n[continuation] data-only control")
    data_only_fit = train_data_only_continuation(
        example_name, seed, baseline_params, train_data, validation_data
    )
    data_only = make_data_only_record(
        data_only_fit,
        baseline_params,
        example_name,
        train_data,
        validation_data,
        test_data,
        tx_eval,
        tx_residual,
        full_W_max,
    )

    # Ungated controls deliberately force all m_max=15 discovered directions.
    print("\n[Stage III] complete library, ungated m=15")
    full_fit = train_stage_f(
        example_name, seed, baseline_params, full_W_max, COMPLETE_LIBRARY,
        train_data, validation_data, "complete ungated m=15"
    )
    full_ungated = make_branch_record(
        "full_ungated", full_fit["params"], baseline_params, example_name,
        train_data, validation_data, test_data, tx_eval, tx_residual,
        full_W_max, COMPLETE_LIBRARY, full_fit["runtime_sec"],
        full_fit["nonfinite_steps"], M_MAX,
        CONFIG["STAGE_F_LAMBDA_SYM_DET"], False,
    )

    print("\n[Stage III] negative control, ungated m=15")
    reduced_fit = train_stage_f(
        example_name, seed, baseline_params, reduced_W_max, REDUCED_LIBRARY,
        train_data, validation_data, "negative control ungated m=15"
    )
    reduced_ungated = make_branch_record(
        "reduced_ungated", reduced_fit["params"], baseline_params, example_name,
        train_data, validation_data, test_data, tx_eval, tx_residual,
        reduced_W_max, REDUCED_LIBRARY, reduced_fit["runtime_sec"],
        reduced_fit["nonfinite_steps"], M_MAX,
        CONFIG["STAGE_F_LAMBDA_SYM_DET"], False,
    )

    # Gated complete-library branch: use the selected prefix, not forced m=15.
    full_gated_fit = None
    if surrogate_full["selected_m"] == 0:
        full_gated = make_exact_abstention_alias(vanilla, "full_gated")
    elif surrogate_full["selected_m"] == M_MAX:
        full_gated = copy.deepcopy(full_ungated)
        full_gated["branch_name"] = "full_gated"
    else:
        selected_m = int(surrogate_full["selected_m"])
        print(f"\n[Stage III] complete library, gated m={selected_m}")
        full_gated_fit = train_stage_f(
            example_name, seed, baseline_params, full_W_selected,
            COMPLETE_LIBRARY, train_data, validation_data,
            f"complete gated m={selected_m}"
        )
        full_gated = make_branch_record(
            "full_gated", full_gated_fit["params"], baseline_params,
            example_name, train_data, validation_data, test_data, tx_eval,
            tx_residual, full_W_selected, COMPLETE_LIBRARY,
            full_gated_fit["runtime_sec"], full_gated_fit["nonfinite_steps"],
            selected_m, CONFIG["STAGE_F_LAMBDA_SYM_DET"], False,
        )

    # Gated negative-control branch.
    reduced_gated_fit = None
    if surrogate_reduced["selected_m"] == 0:
        reduced_gated = make_exact_abstention_alias(vanilla, "reduced_gated")
    elif surrogate_reduced["selected_m"] == M_MAX:
        reduced_gated = copy.deepcopy(reduced_ungated)
        reduced_gated["branch_name"] = "reduced_gated"
    else:
        selected_m = int(surrogate_reduced["selected_m"])
        print(f"\n[Stage III] negative control, gated m={selected_m}")
        reduced_gated_fit = train_stage_f(
            example_name, seed, baseline_params, reduced_W_selected,
            REDUCED_LIBRARY, train_data, validation_data,
            f"negative control gated m={selected_m}"
        )
        reduced_gated = make_branch_record(
            "reduced_gated", reduced_gated_fit["params"], baseline_params,
            example_name, train_data, validation_data, test_data, tx_eval,
            tx_residual, reduced_W_selected, REDUCED_LIBRARY,
            reduced_gated_fit["runtime_sec"],
            reduced_gated_fit["nonfinite_steps"], selected_m,
            CONFIG["STAGE_F_LAMBDA_SYM_DET"], False,
        )

    # The mu-selected branch also uses its adaptively selected prefix. If no
    # mu candidate passes, it is an exact Stage-I abstention alias.
    mu_fit = None
    selected_mu_m = int(mu_sweep[selected_mu]["selected_m"])
    if selected_mu_m == 0:
        mu_selected = make_exact_abstention_alias(vanilla, "mu_selected")
    else:
        print(
            f"\n[Stage III] validation-selected mu={selected_mu:+.2f}, "
            f"m={selected_mu_m}"
        )
        mu_fit = train_stage_f(
            example_name, seed, baseline_params, selected_mu_W,
            MU_SWEEP_LIBRARIES[selected_mu], train_data, validation_data,
            f"selected mu {selected_mu:+.2f}, m={selected_mu_m}"
        )
        mu_selected = make_branch_record(
            "mu_selected", mu_fit["params"], baseline_params, example_name,
            train_data, validation_data, test_data, tx_eval, tx_residual,
            selected_mu_W, MU_SWEEP_LIBRARIES[selected_mu],
            mu_fit["runtime_sec"], mu_fit["nonfinite_steps"], selected_mu_m,
            CONFIG["STAGE_F_LAMBDA_SYM_DET"], False,
        )

    result = {
        "example_name": example_name,
        "example_label": spec["label"],
        "seed": int(seed),
        "data": {
            "n_trajectories": int(CONFIG["N_TRAJECTORIES"]),
            "n_train_trajectories": int(data["train_paths"].shape[0]),
            "n_validation_trajectories": int(data["validation_paths"].shape[0]),
            "n_test_trajectories": int(data["test_paths"].shape[0]),
            "t_final": float(spec["t_final"]),
            "dt": float(spec["dt"]),
            "x_box": tuple(float(v) for v in spec["x_box"]),
        },
        "stage_i": {
            "best_epoch": int(stage_i["best_epoch"]),
            "best_validation_nll": float(stage_i["best_validation_nll"]),
            "runtime_sec": float(stage_i["runtime_sec"]),
            "nonfinite_steps": int(stage_i["nonfinite_steps"]),
        },
        "spectral": {
            "oracle": {"full": oracle_full, "reduced": oracle_reduced},
            "surrogate": {"full": surrogate_full, "reduced": surrogate_reduced},
        },
        "mu_sweep": mu_sweep,
        "selected_mu": float(selected_mu),
        "branches": {
            "vanilla": vanilla,
            "data_only_continuation": data_only,
            "full_ungated": full_ungated,
            "full_gated": full_gated,
            "reduced_ungated": reduced_ungated,
            "reduced_gated": reduced_gated,
            "mu_selected": mu_selected,
        },
    }

    del data, baseline_params, data_only_fit, full_fit, reduced_fit
    if full_gated_fit is not None:
        del full_gated_fit
    if reduced_gated_fit is not None:
        del reduced_gated_fit
    if mu_fit is not None:
        del mu_fit
    gc.collect()
    return result


E6_RESULTS = []
for example_name in CONFIG["E6_EXAMPLES"]:
    E6_RESULTS.append(run_one_example(example_name, CONFIG["SEED"]))
    try:
        path = Path(CONFIG["E6_RESULTS_PATH"])
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "wb") as handle:
            pickle.dump(E6_RESULTS, handle)
        print("Saved partial numerical results to", path)
    except OSError as exc:
        print("Warning: could not save partial E6 results:", exc)

print("\nCompleted E6 examples:", [r["example_name"] for r in E6_RESULTS])

## Cell 7 — corrected rank, adaptive-m, and E6 final report

In [ ]:
# ============================================================================
# Cell 7 — corrected rank, adaptive-m, and E6 final report
# ============================================================================

if "E6_RESULTS" not in globals() or not E6_RESULTS:
    raise RuntimeError("E6_RESULTS is empty. Run Cells 1–6 first.")

results = list(E6_RESULTS)
expected_examples = list(CONFIG["E6_EXAMPLES"])
observed_examples = [r["example_name"] for r in results]
if observed_examples != expected_examples:
    raise RuntimeError(
        f"Expected results in order {expected_examples}, obtained {observed_examples}."
    )

branch_order = [
    "vanilla",
    "data_only_continuation",
    "full_ungated",
    "full_gated",
    "reduced_ungated",
    "reduced_gated",
    "mu_selected",
]
branch_labels = {
    "vanilla": "Vanilla Stage I",
    "data_only_continuation": "Data-only continuation",
    "full_ungated": "Complete, ungated m=15",
    "full_gated": "Complete, adaptive gated",
    "reduced_ungated": "Control, ungated m=15",
    "reduced_gated": "Control, adaptive gated",
    "mu_selected": "Validation-selected mu",
}


def pct_reduction(baseline, value):
    return (
        100.0
        * (float(baseline) - float(value))
        / max(abs(float(baseline)), 1.0e-30)
    )


def fmt(x, precision=4):
    return f"{float(x):.{precision}e}"


print("=" * 158)
print("FINAL E6 REPORT — THREE 20D EXAMPLES, ONE SEED, m_max=15")
print("=" * 158)
print("Seed                                    :", CONFIG["SEED"])
print("Maximum discovered generators m_max     :", M_MAX)
print("Complete-library size / expected rank    :",
      COMPLETE_LIBRARY["size"], "/", COMPLETE_LIBRARY["expected_rank"])
print("Negative-control size / expected rank    :",
      REDUCED_LIBRARY["size"], "/", REDUCED_LIBRARY["expected_rank"])
print("Single-mu size / expected rank           :",
      next(iter(MU_SWEEP_LIBRARIES.values()))["size"], "/",
      next(iter(MU_SWEEP_LIBRARIES.values()))["expected_rank"])
print("Bootstrap rediscoveries                  :", CONFIG["E6_BOOTSTRAP_REPEATS"])
print("Complete mu grid                         :", CONFIG["COMPLETE_MU_LIST"])
print("Swept mu values                          :", CONFIG["MU_SWEEP_LIST"])

summary = {}

for result in results:
    label = result["example_label"]
    print("\n" + "#" * 158)
    print(label)
    print("#" * 158)

    print("\n[1] DATA AND TRAINING SETUP")
    data = result["data"]
    print(
        f"trajectories train/validation/test = "
        f"{data['n_train_trajectories']}/"
        f"{data['n_validation_trajectories']}/"
        f"{data['n_test_trajectories']} | "
        f"T={data['t_final']} | dt={data['dt']} | x-box={data['x_box']}"
    )
    print(
        f"Stage-I best epoch={result['stage_i']['best_epoch']} | "
        f"validation NLL={result['stage_i']['best_validation_nll']:.6e} | "
        f"runtime={result['stage_i']['runtime_sec']:.2f}s"
    )

    print("\n[2] NORMALIZED-GRAM RANK AND ADAPTIVE ADEQUACY DIAGNOSTIC")
    print(
        f"{'operator':<11s} {'library':<9s} {'R/rank/exp':>13s} "
        f"{'m*':>3s} {'mean q@15':>12s} {'mean q@m*':>12s} "
        f"{'null ratio':>11s} {'boot max':>10s} {'decision':>10s}"
    )
    print("-" * 116)

    for operator_name in ("oracle", "surrogate"):
        for library_name in ("full", "reduced"):
            diag = result["spectral"][operator_name][library_name]
            selected_score = diag["selected_validation_aggregate_score"]
            selected_text = (
                fmt(selected_score)
                if np.isfinite(selected_score)
                else "n/a"
            )
            print(
                f"{operator_name:<11s} {library_name:<9s} "
                f"{diag['r_raw']:3d}/{diag['r_effective']:3d}/"
                f"{diag['expected_rank']:<3d} "
                f"{diag['selected_m']:3d} "
                f"{fmt(diag['validation_aggregate_score']):>12s} "
                f"{selected_text:>12s} "
                f"{diag['score_to_null']:11.3e} "
                f"{diag['bootstrap_max_angle_deg']:10.2f} "
                f"{diag['decision']:>10s}"
            )
            if not diag["rank_verified"]:
                print(" " * 13 + "RANK CHECK FAILED")
            if operator_name == "surrogate":
                print(" " * 13 + f"reason: {diag['reason']}")

    oracle_full = result["spectral"]["oracle"]["full"][
        "validation_aggregate_score"
    ]
    oracle_reduced = result["spectral"]["oracle"]["reduced"][
        "validation_aggregate_score"
    ]
    surrogate_full = result["spectral"]["surrogate"]["full"][
        "validation_aggregate_score"
    ]
    surrogate_reduced = result["spectral"]["surrogate"]["reduced"][
        "validation_aggregate_score"
    ]
    print(
        "oracle control/complete fixed-m_max score ratio   : "
        f"{oracle_reduced / max(oracle_full, 1e-30):.6e}"
    )
    print(
        "surrogate control/complete fixed-m_max score ratio: "
        f"{surrogate_reduced / max(surrogate_full, 1e-30):.6e}"
    )

    print(
        "\nComplete-library held-out Rayleigh scores of "
        "train-discovered modes 1:15:"
    )
    print(np.array2string(
        np.asarray(
            result["spectral"]["surrogate"]["full"]["validation_scores"]
        ),
        precision=5,
        separator=", ",
    ))
    print(
        "Negative-control held-out Rayleigh scores of "
        "train-discovered modes 1:15:"
    )
    print(np.array2string(
        np.asarray(
            result["spectral"]["surrogate"]["reduced"]["validation_scores"]
        ),
        precision=5,
        separator=", ",
    ))

    print("\n[3] MU SWEEP — FROZEN STAGE-I SURROGATE")
    print(
        f"{'mu':>7s} {'R/rank':>9s} {'m*':>3s} "
        f"{'mean q@15':>12s} {'mean q@m*':>12s} "
        f"{'null ratio':>11s} {'boot max':>10s} {'decision':>10s}"
    )
    print("-" * 92)
    for mu in CONFIG["MU_SWEEP_LIST"]:
        diag = result["mu_sweep"][float(mu)]
        marker = "*" if np.isclose(float(mu), result["selected_mu"]) else " "
        selected_score = diag["selected_validation_aggregate_score"]
        selected_text = (
            fmt(selected_score)
            if np.isfinite(selected_score)
            else "n/a"
        )
        print(
            f"{float(mu):+6.2f}{marker} "
            f"{diag['r_raw']:3d}/{diag['r_effective']:<3d} "
            f"{diag['selected_m']:3d} "
            f"{fmt(diag['validation_aggregate_score']):>12s} "
            f"{selected_text:>12s} "
            f"{diag['score_to_null']:11.3e} "
            f"{diag['bootstrap_max_angle_deg']:10.2f} "
            f"{diag['decision']:>10s}"
        )
    print(
        "* validation-selected mu =",
        result["selected_mu"],
        "| selected m =",
        result["mu_sweep"][result["selected_mu"]]["selected_m"],
    )

    print("\n[4] DOWNSTREAM RESULTS")
    print(
        f"{'branch':<29s} {'m':>3s} {'val NLL':>13s} {'test NLL':>13s} "
        f"{'drift MSE':>13s} {'sigma MSE':>13s} {'det after':>13s} "
        f"{'drift red.':>11s} {'sigma red.':>11s} {'abstain':>9s}"
    )
    print("-" * 146)
    baseline = result["branches"]["vanilla"]
    for branch_name in branch_order:
        branch = result["branches"][branch_name]
        print(
            f"{branch_labels[branch_name]:<29s} "
            f"{branch['effective_m']:3d} "
            f"{fmt(branch['validation_nll']):>13s} "
            f"{fmt(branch['test_nll']):>13s} "
            f"{fmt(branch['drift_mse']):>13s} "
            f"{fmt(branch['sigma_mse']):>13s} "
            f"{fmt(branch['det_residual_after']):>13s} "
            f"{pct_reduction(baseline['drift_mse'], branch['drift_mse']):10.2f}% "
            f"{pct_reduction(baseline['sigma_mse'], branch['sigma_mse']):10.2f}% "
            f"{str(branch['abstained']):>9s}"
        )

    print("\n[5] ABSTENTION IDENTITY AUDIT")
    for branch_name in ("full_gated", "reduced_gated", "mu_selected"):
        branch = result["branches"][branch_name]
        if branch["abstained"]:
            status = (
                "PASS"
                if branch["param_max_abs_diff_vs_baseline"]
                <= float(CONFIG["E6_MAX_ABSTENTION_IDENTITY_ERROR"])
                else "FAIL"
            )
            print(
                f"{branch_labels[branch_name]}: abstained; "
                f"max parameter difference="
                f"{branch['param_max_abs_diff_vs_baseline']:.3e} [{status}]"
            )
        else:
            print(
                f"{branch_labels[branch_name]}: used "
                f"m={branch['effective_m']}; identity audit not applicable."
            )

    summary[result["example_name"]] = {
        "selected_mu": float(result["selected_mu"]),
        "complete_selected_m": int(
            result["spectral"]["surrogate"]["full"]["selected_m"]
        ),
        "control_selected_m": int(
            result["spectral"]["surrogate"]["reduced"]["selected_m"]
        ),
        "complete_decision": result["spectral"]["surrogate"]["full"][
            "decision"
        ],
        "control_decision": result["spectral"]["surrogate"]["reduced"][
            "decision"
        ],
        "complete_score_mmax": float(surrogate_full),
        "control_score_mmax": float(surrogate_reduced),
        "score_ratio": float(
            surrogate_reduced / max(surrogate_full, 1.0e-30)
        ),
        "vanilla_drift_mse": float(baseline["drift_mse"]),
        "vanilla_sigma_mse": float(baseline["sigma_mse"]),
        "full_gated_drift_mse": float(
            result["branches"]["full_gated"]["drift_mse"]
        ),
        "full_gated_sigma_mse": float(
            result["branches"]["full_gated"]["sigma_mse"]
        ),
        "control_gated_drift_mse": float(
            result["branches"]["reduced_gated"]["drift_mse"]
        ),
        "control_gated_sigma_mse": float(
            result["branches"]["reduced_gated"]["sigma_mse"]
        ),
    }


print("\n" + "=" * 158)
print("COMPACT THREE-EXAMPLE SUMMARY")
print("=" * 158)
print(
    f"{'example':<38s} {'rank full/control':>19s} "
    f"{'m* full/control':>17s} {'q15 full/control':>23s} "
    f"{'selected mu':>12s} {'drift vanilla/full/control':>31s} "
    f"{'sigma vanilla/full/control':>31s}"
)
print("-" * 180)

for result in results:
    s = summary[result["example_name"]]
    full_diag = result["spectral"]["surrogate"]["full"]
    control_diag = result["spectral"]["surrogate"]["reduced"]
    print(
        f"{result['example_label']:<38s} "
        f"{full_diag['r_effective']:3d}/{control_diag['r_effective']:<3d} "
        f"{s['complete_selected_m']:3d}/{s['control_selected_m']:<3d} "
        f"{s['complete_score_mmax']:.3e}/{s['control_score_mmax']:.3e} "
        f"{s['selected_mu']:+11.2f} "
        f"{s['vanilla_drift_mse']:.3e}/"
        f"{s['full_gated_drift_mse']:.3e}/"
        f"{s['control_gated_drift_mse']:.3e} "
        f"{s['vanilla_sigma_mse']:.3e}/"
        f"{s['full_gated_sigma_mse']:.3e}/"
        f"{s['control_gated_sigma_mse']:.3e}"
    )

rank_safe = all(
    r["spectral"][operator][library]["rank_verified"]
    for r in results
    for operator in ("oracle", "surrogate")
    for library in ("full", "reduced")
)
complete_better = all(v["score_ratio"] > 1.0 for v in summary.values())
control_abstains = all(
    v["control_decision"] == "ABSTAIN" for v in summary.values()
)
identity_safe = all(
    (not r["branches"][branch]["abstained"])
    or r["branches"][branch]["param_max_abs_diff_vs_baseline"]
        <= float(CONFIG["E6_MAX_ABSTENTION_IDENTITY_ERROR"])
    for r in results
    for branch in ("full_gated", "reduced_gated", "mu_selected")
)

print("\nDirectional checks (single-seed evidence only):")
print("  all normalized-Gram rank checks pass             :", rank_safe)
print("  complete fixed-m_max score lower on all examples :", complete_better)
print("  negative control abstains on all examples        :", control_abstains)
print("  every actual abstention is an exact Stage-I alias:", identity_safe)
print(
    "\nBecause this notebook intentionally uses one seed, these are "
    "directional checks, not confidence intervals."
)

E6_FINAL_SUMMARY = {
    "seed": int(CONFIG["SEED"]),
    "m_max": int(M_MAX),
    "complete_library_size": int(COMPLETE_LIBRARY["size"]),
    "complete_expected_rank": int(COMPLETE_LIBRARY["expected_rank"]),
    "control_library_size": int(REDUCED_LIBRARY["size"]),
    "control_expected_rank": int(REDUCED_LIBRARY["expected_rank"]),
    "bootstrap_repeats": int(CONFIG["E6_BOOTSTRAP_REPEATS"]),
    "per_example": summary,
    "all_rank_checks_pass": bool(rank_safe),
    "complete_better_all_examples": bool(complete_better),
    "control_abstains_all_examples": bool(control_abstains),
    "abstention_identity_safe": bool(identity_safe),
}

print("\nE6_FINAL_SUMMARY =")
for key, value in E6_FINAL_SUMMARY.items():
    if key != "per_example":
        print(f"  {key}: {value}")
